# Maths and statistics

> Elementary and special functions, complex numbers, statistical reductions and tolerant comparisons.

In [1]:
import rustymath as rm

## Elementary functions

Functions accept scalars or arrays. Integer inputs to float functions compute at an appropriate float width. Float arrays retain their width:

In [2]:
rm.exp([0., 1, 2]), rm.log([1., 10, 100]), rm.sin([0., 0.5, 1])

(Array([1.0, 2.7182818, 7.3890561], dtype=f64),
 Array([0.0, 2.3025851, 4.6051702], dtype=f64),
 Array([0.0, 0.47942554, 0.84147098], dtype=f64))

Functions that NumPy lacks keep their own names. `root(n, y)` gives the `n`th root of `y`:

In [3]:
rm.root(3, [8., 27., 1000.])

Array([2.0, 3.0, 10.0], dtype=f64)

`round(x, decimals)` rounds values to a decimal place, with ties to even. `rint` rounds to whole values. Neither changes how many digits the array's `repr` displays. `degrees` and `radians` convert angular units.

In [4]:
rm.round([1.2345, -2.71828], 2), rm.degrees([0., 1.5707963267948966])

(Array([1.23, -2.72], dtype=f64), Array([0.0, 90.0], dtype=f64))

The Rust names follow Rust's standard library instead. Python's `log`, `power`, `arcsin` and `arctan2` are Rust's `ln`, `pow`, `asin` and `atan2`.

## Complex numbers

A complex array gives complex results, at each function's principal value. For `exp`, `log`, `power`, `sin`, `cos`, `sinh` and `cosh`, rustymath computes a complex array through its vectorized real loops:

In [5]:
rm.sqrt([-4+0j, 3+4j]), rm.log([-1+0j])

(Array([2j, (2+1j)], dtype=c128), Array([3.1415927j], dtype=c128))

## Special functions

The special functions take SciPy's names, such as `gamma`, `gammaln`, `erf`, `erfinv`, `betainc` and `lambertw`:

In [6]:
rm.gamma([0.5, 5, 10]), rm.erfinv(0.5)

(Array([1.7724539, 24.0, 362880.0], dtype=f64), 0.4769362762044699)

## Reductions and statistics

With no `axis`, reductions summarize all items. Pass an axis position, name or tuple of axes to summarize those axes. Here each row holds one series. Clipping bounds the original values in one pass:

In [7]:
measurements = rm.array([[1., 2, 3], [4, 5, 6]])
rm.mean(measurements, axis=1), rm.std(measurements, axis=1), rm.clip(measurements, 2., 5.)

(Array([2.0, 5.0], dtype=f64),
 Array([0.81649658, 0.81649658], dtype=f64),
 Array([[2.0, 2.0, 3.0], [4.0, 5.0, 5.0]], dtype=f64))

`diff` takes adjacent differences along an axis; `ptp` gives maximum minus minimum, and `trace` sums the diagonal of each matrix in the last two axes. Integer differences check overflow rather than wrapping. Boolean differences report a change (XOR).

In [8]:
rm.diff(measurements), rm.ptp(measurements, axis=1), rm.trace(measurements)

(Array([[1.0, 1.0], [1.0, 1.0]], dtype=f64), Array([2.0, 2.0], dtype=f64), 6.0)

`var` and `std` divide by `n - ddof`, with `ddof=0` by default. `average` accepts weights under the same leading-axis broadcasting rule as other functions. `cov` and `corrcoef` treat rows as variables unless `rowvar=False`. Float16 summaries accumulate and return float32; complex variance and norms return real values:

In [9]:
rm.cov(measurements), rm.corrcoef(measurements)

(Array([[1.0, 1.0], [1.0, 1.0]], dtype=f64),
 Array([[1.0, 1.0], [1.0, 1.0]], dtype=f64))

`norm` computes an elementwise p-norm over the selected axes. It does not switch to a matrix-induced norm for two-dimensional input:

In [10]:
rm.norm([3., 4.]), rm.isfinite([1., float('inf'), float('nan')])

(5.0, Array([True, False, False], dtype=bool))

Scans keep a running result. `logical_scan` takes any Boolean dyad as a 2 by 2 truth table. This table computes running exclusive-or:

In [11]:
rm.logical_scan([False, True, True, False], [[False, True], [True, False]])

Array([False, True, False, False], dtype=bool)

## Tolerant comparison

APL compares floats within a relative tolerance. Two floats are equal when their difference is less than the tolerance times the larger magnitude. With the usual tolerance, `0.1 + 0.2` equals `0.3`. `rm.equal`, `not_equal`, `less`, `less_equal`, `greater` and `greater_equal` compare this way. Their tolerance `ct` defaults to `rm.CT`, 1e-14. Pass `ct=0` to compare exactly:

In [12]:
rm.equal(0.1 + 0.2, 0.3), rm.equal(0.1 + 0.2, 0.3, ct=0)

(True, False)

`floor`, `ceil`, `remainder`, `gcd` and `lcm` take `ct` too. A value within tolerance of a whole number counts as that whole number:

In [13]:
rm.floor([2.9999999999999996, 2.5])

Array([3.0, 2.0], dtype=f64)

`remainder` takes its arguments in NumPy's order. `rm.remainder(x, y)` is the Rust `residue(y, x)`.

## Checking accuracy

In [14]:
import numpy as np

To check the accuracy yourself, compare with NumPy. `np.spacing` gives the size of one ulp at each value. Dividing the difference by it gives the distance between the two results in ulps:

In [15]:
xs = np.linspace(-10, 10, 100_001)
y = np.exp(xs)
(np.abs(rm.exp(xs) - y) / np.spacing(y)).max()

np.float64(2.0)